<a href="https://colab.research.google.com/github/mateosanlu/DeepLearning/blob/main/Deep_Learning_Actividad_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller Actividad 1 - Redes Neuronales: Clasificador de Créditos

##Mateo Sandoval L

Este cuaderno implementa el diseño, análisis matemático y ejecución de una arquitectura de red neuronal artificial aplicada a la evaluación automática de solicitudes de crédito financiero.


### Importar las librerías
Cargamos la librería `numpy` para gestionar operaciones algebraicas y matriciales de alto rendimiento de manera vectorizada.


In [15]:
import numpy as np

print("NumPy está listo para el procesamiento matricial")


NumPy está listo para el procesamiento matricial


## 1. Crear matriz de características (Inputs)
Definimos la matriz de datos de entrada X. En este modelo, las filas representan a los **clientes (muestras)** y las columnas representan sus **atributos financieros (características)**, los cuales han sido previamente normalizados en un rango de 0 a 1:
* **Columna 1:** Historial Crediticio.
* **Columna 2:** Relación Ingresos/Deuda.
* **Columna 3:** Capacidad de Ahorro Mensual.


In [5]:
X = np.array([
    [0.85, 0.75, 0.90],  # Cliente 1: Excelente perfil financiero
    [0.20, 0.30, 0.15],  # Cliente 2: Riesgo financiero alto
    [0.60, 0.55, 0.50],  # Cliente 3: Perfil intermedio / moderado
    [0.45, 0.40, 0.65]   # Cliente 4: Perfil de riesgo medio-bajo
])

print("Matriz de Clientes (X):\n", X)
print("Forma de X (Clientes x Atributos):", X.shape)


Matriz de Clientes (X):
 [[0.85 0.75 0.9 ]
 [0.2  0.3  0.15]
 [0.6  0.55 0.5 ]
 [0.45 0.4  0.65]]
Forma de X (Clientes x Atributos): (4, 3)


## 2. Definir parámetros del Perceptrón Simple
Establecemos los pesos W y el término de sesgo (*bias*).
* Los pesos determinan el grado de importancia de cada variable. En este diseño, el **Historial Crediticio (0.6)** es el factor más determinante.
* El *bias* (-0.4) actúa como el umbral mínimo exigido por la política de riesgo del banco.


In [6]:
W = np.array([0.6, 0.3, 0.1])
bias = -0.4

print("\nPesos iniciales (W):", W)
print("Sesgo (Bias):", bias)



Pesos iniciales (W): [0.6 0.3 0.1]
Sesgo (Bias): -0.4


## 3. Combinación lineal (Cálculo Neto)
Calculamos la suma ponderada neta mediante la multiplicación matricial (operador `@`) entre los datos de entrada y los pesos, sumando el sesgo en una sola operación vectorizada:

$$Z = X \cdot W + b$$

Este paso proyecta las características de los clientes en una única calificación continua de solvencia.


In [7]:
Z = X @ W + bias

print("\nResultado antes de la activación (Z):")
print(Z)



Resultado antes de la activación (Z):
[ 0.425 -0.175  0.175  0.055]


## 4. Activación de la Función Escalón (Inferencia Simple)
Aplicamos la función de activación. Si la calificación neta $Z \ge 0$, el crédito es pre-aprobado (1); de lo contrario, es denegado (0).

Esta operación simula el comportamiento de un **Perceptrón Simple**, un clasificador lineal binario rígido.


In [9]:
def funcion_escalon(z):
    return np.where(z >= 0, 1, 0)

prediccion = funcion_escalon(Z)
print("\nPre-aprobación del Perceptrón Simple (1=Aprobado, 0=Rechazado):")
print(prediccion)



Pre-aprobación del Perceptrón Simple (1=Aprobado, 0=Rechazado):
[1 0 1 1]


## 5. Arquitectura Multicapa: Pesos para Capa Oculta
Para capturar relaciones financieras más complejas y no lineales (que un perceptrón simple no puede resolver), escalamos el modelo a una **Red Neuronal Multicapa (MLP)**.

Definimos una matriz de pesos $W_1$ de dimensiones ($3 \times 3$) y un vector de sesgos $b_1$. Esto significa que pasamos de una sola neurona a una **capa oculta con 3 neuronas artificiales**, donde cada una se especializará en evaluar un perfil de riesgo combinado distinto.


In [11]:
W1 = np.array([
    [0.5, -0.2, 0.4],
    [0.3, 0.6, -0.1],
    [-0.2, 0.4, 0.5]
])
b1 = np.array([-0.1, -0.2, 0.0])

Z1 = X @ W1 + b1
print("\nCombinación Lineal Capa Oculta (Z1):\n", Z1)
print("Forma de Z1:", Z1.shape)



Combinación Lineal Capa Oculta (Z1):
 [[ 3.70000000e-01  4.40000000e-01  7.15000000e-01]
 [ 6.00000000e-02 -2.77555756e-17  1.25000000e-01]
 [ 2.65000000e-01  2.10000000e-01  4.35000000e-01]
 [ 1.15000000e-01  2.10000000e-01  4.65000000e-01]]
Forma de Z1: (4, 3)


## 6. Regla de Activación de la Capa: ReLU
Aplicamos la función de activación **ReLU** (Unidad Lineal Rectificada): $f(z) = \max(0, z)$.

La función ReLU introduce no-linealidad en la red y descarta los valores negativos convirtiéndolos en cero. En nuestra temática, modela la lógica donde los indicadores financieros negativos representan una ausencia total de viabilidad de éxito y se neutralizan antes de la decisión final.


In [12]:
def relu(z):
    return np.maximum(0, z)

A1 = relu(Z1)
print("\nActivación Capa Oculta (A1):\n", A1)



Activación Capa Oculta (A1):
 [[0.37  0.44  0.715]
 [0.06  0.    0.125]
 [0.265 0.21  0.435]
 [0.115 0.21  0.465]]


## 7. Capa de Salida (Consolidación de Criterios)
Definimos la matriz de pesos $W_2$ y el sesgo $b_2$ para la capa de salida. Esta etapa recopila las abstracciones generadas por la capa oculta ($A_1$) y las comprime linealmente en un único valor numérico por cliente.


In [13]:
W2 = np.array([
    [0.8],
    [0.7],
    [-0.2]
])
b2 = np.array([-0.3])

Z2 = A1 @ W2 + b2
print("\nCombinación Lineal de Salida (Z2):\n", Z2)



Combinación Lineal de Salida (Z2):
 [[ 0.161]
 [-0.277]
 [-0.028]
 [-0.154]]


## 8. Inferencia Probabilística (Función Sigmoide) y Dictamen Final
Para obtener una salida interpretable en términos financieros, mapeamos el valor $Z_2$ a través de la **función de activación Sigmoide**:

$$S(z) = \frac{1}{1 + e^{-z}}$$

* La función sigmoide comprime cualquier valor real al rango abierto $(0, 1)$, permitiendo interpretar el resultado como la **probabilidad de éxito de pago del crédito**.
* Finalmente, aplicamos un umbral estricto del 50% ($\ge 0.5$) para emitir el dictamen final definitivo.


In [14]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

A2 = sigmoid(Z2)
print("\nProbabilidad final de aprobación (0 a 1):")
print(A2)

prediccion_final = np.where(A2 >= 0.5, 1, 0)
print("\nDictamen Final Estricto (1=Crédito Otorgado, 0=Denegado):")
print(prediccion_final)



Probabilidad final de aprobación (0 a 1):
[[0.54016328]
 [0.43118942]
 [0.49300046]
 [0.46157591]]

Dictamen Final Estricto (1=Crédito Otorgado, 0=Denegado):
[[1]
 [0]
 [0]
 [0]]


### Conclusión del Análisis para el Informe
Al comparar ambos procesos, el **Perceptrón Simple** (Paso 4) arrojaba una pre-aprobación permisiva debido a su naturaleza puramente lineal. Al procesar los mismos clientes a través de la **Red Neuronal Multicapa** con activaciones ReLU y Sigmoide (Paso 8), el modelo se vuelve mucho más riguroso y selectivo, filtrando los perfiles marginales y otorgando la aprobación definitiva únicamente al **Cliente 1**, quien cuenta con las métricas de solvencia más sólidas y consistentes.
